# DESCARGAR DATOS TROPOMI TENERIFE, mediante conexión con Microsoft Planetary Computer

In [ ]:
!pip install pystac-client planetary-computer xarray netcdf4
!pip install contextily
!pip install netcdf4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 955.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pystac_client
import planetary_computer
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import contextily as cx
import requests
import os
import warnings
import scipy.ndimage as ndimage


In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2023-12-01"
FECHA_FIN = "2023-12-31"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2023-12-01 al 2023-12-31

📅 Analizando: 2023-12-01
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-02
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-03
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2023-12-04
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-05
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-06
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-07
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-08
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2023-12-09
    ✅ Éxito: 16 píxeles guardados.

📅 Analizando: 2023-12-10
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-11
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2023-12-12
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2023-12-13
    ✅ Éxito: 

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2022-01-01"
FECHA_FIN = "2022-01-31"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2022-01-01 al 2022-01-31

📅 Analizando: 2022-01-01
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-01-02
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-01-03
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-01-04
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-01-05
    ✅ Éxito: 4 píxeles guardados.

📅 Analizando: 2022-01-06
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-01-07
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-01-08
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-01-09
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-01-10
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-01-11
    ✅ Éxito: 5 píxeles guardados.

📅 Analizando: 2022-01-12
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-01-13
    ✅ Éxito: 2 píxeles guardados.

📅 Ana

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2022-02-01"
FECHA_FIN = "2022-02-27"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2022-02-01 al 2022-02-27

📅 Analizando: 2022-02-01
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2022-02-02
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2022-02-03
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2022-02-04
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2022-02-05
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2022-02-06
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2022-02-07
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2022-02-08
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-02-09
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-02-10
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-02-11
    ✅ Éxito: 7 píxeles guardados.

📅 Analizando: 2022-02-12
    ✅ Éxito: 4 píxeles guardados.

📅 Analizando: 2022-02-13
    ✅ Éxito: 1 píxeles gua

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2022-03-01"
FECHA_FIN = "2022-03-31"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2022-03-01 al 2022-03-31

📅 Analizando: 2022-03-01
    ✅ Éxito: 5 píxeles guardados.

📅 Analizando: 2022-03-02
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-03-03
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-03-04
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-03-05
    ✅ Éxito: 3 píxeles guardados.

📅 Analizando: 2022-03-06
    ✅ Éxito: 3 píxeles guardados.

📅 Analizando: 2022-03-07
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-03-08
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-03-09
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-03-10
    ✅ Éxito: 6 píxeles guardados.

📅 Analizando: 2022-03-11
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-03-12
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-03-13
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Ana

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2022-04-01"
FECHA_FIN = "2022-04-30"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2022-04-01 al 2022-04-30

📅 Analizando: 2022-04-01
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-04-02
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-04-03
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-04-04
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-04-05
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-04-06
    ✅ Éxito: 22 píxeles guardados.

📅 Analizando: 2022-04-07
    ✅ Éxito: 5 píxeles guardados.

📅 Analizando: 2022-04-08
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-04-09
    ✅ Éxito: 4 píxeles guardados.

📅 Analizando: 2022-04-10
    ✅ Éxito: 3 píxeles guardados.

📅 Analizando: 2022-04-11
    ✅ Éxito: 24 píxeles guardados.

📅 Analizando: 2022-04-12
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-04-13
    ⚠️ Día descartado: Todo nubes (NaN).

📅 A

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2022-05-01"
FECHA_FIN = "2022-05-31"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2022-05-01 al 2022-05-31

📅 Analizando: 2022-05-01
    ✅ Éxito: 8 píxeles guardados.

📅 Analizando: 2022-05-02
    ✅ Éxito: 56 píxeles guardados.

📅 Analizando: 2022-05-03
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-05-04
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-05-05
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2022-05-06
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-05-07
    ✅ Éxito: 5 píxeles guardados.

📅 Analizando: 2022-05-08
    ✅ Éxito: 17 píxeles guardados.

📅 Analizando: 2022-05-09
    ✅ Éxito: 5 píxeles guardados.

📅 Analizando: 2022-05-10
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2022-05-11
    ✅ Éxito: 2 píxeles guardados.

📅 Analizando: 2022-05-12
    ✅ Éxito: 3 píxeles guardados.

📅 Analizando: 2022-05-13
    ✅ Éxito: 8 píxeles guardados.

📅 Analizando: 202

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2024-08-01"
FECHA_FIN = "2024-08-31"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2024-08-01 al 2024-08-31

📅 Analizando: 2024-08-01
    ✅ Éxito: 57 píxeles guardados.

📅 Analizando: 2024-08-02
    ✅ Éxito: 5 píxeles guardados.

📅 Analizando: 2024-08-03
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-08-04
    ✅ Éxito: 8 píxeles guardados.

📅 Analizando: 2024-08-05
    ✅ Éxito: 14 píxeles guardados.

📅 Analizando: 2024-08-06
    ❌ Error procesando el día: The request exceeded the maximum allowed time, please try again. If the issue persists, please contact planetarycomputer@microsoft.com.



📅 Analizando: 2024-08-07
    ✅ Éxito: 10 píxeles guardados.

📅 Analizando: 2024-08-08
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-08-09
    ✅ Éxito: 9 píxeles guardados.

📅 Analizando: 2024-08-10
    ✅ Éxito: 20 píxeles guardados.

📅 Analizando: 2024-08-11
    ✅ Éxito: 88 píxeles guardados.

📅 Anal

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2024-09-01"
FECHA_FIN = "2024-09-30"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2024-09-01 al 2024-09-30

📅 Analizando: 2024-09-01
    ✅ Éxito: 15 píxeles guardados.

📅 Analizando: 2024-09-02
    ✅ Éxito: 13 píxeles guardados.

📅 Analizando: 2024-09-03
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-09-04
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-09-05
    ✅ Éxito: 3 píxeles guardados.

📅 Analizando: 2024-09-06
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2024-09-07
    ✅ Éxito: 22 píxeles guardados.

📅 Analizando: 2024-09-08
    ✅ Éxito: 3 píxeles guardados.

📅 Analizando: 2024-09-09
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-09-10
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2024-09-11
    ✅ Éxito: 10 píxeles guardados.

📅 Analizando: 2024-09-12
    ✅ Éxito: 9 píxeles guardados.

📅 Analizando: 2024-09-13
    ✅ Éxito: 11 píxeles guardados.

📅 Analiza

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2024-10-01"
FECHA_FIN = "2024-10-31"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2024-10-01 al 2024-10-31

📅 Analizando: 2024-10-01
    ✅ Éxito: 4 píxeles guardados.

📅 Analizando: 2024-10-02
    ✅ Éxito: 10 píxeles guardados.

📅 Analizando: 2024-10-03
    ✅ Éxito: 18 píxeles guardados.

📅 Analizando: 2024-10-04
    ✅ Éxito: 14 píxeles guardados.

📅 Analizando: 2024-10-05
    ✅ Éxito: 8 píxeles guardados.

📅 Analizando: 2024-10-06
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-10-07
    ✅ Éxito: 3 píxeles guardados.

📅 Analizando: 2024-10-08
    ✅ Éxito: 1 píxeles guardados.

📅 Analizando: 2024-10-09
    ✅ Éxito: 20 píxeles guardados.

📅 Analizando: 2024-10-10
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-10-11
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-10-12
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-10-13
    ✅ Éxito: 3 píxeles guardados.

📅 Analiz

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# ==============================================================================
# 0. CONECTAR A GOOGLE DRIVE
# ==============================================================================
drive.mount('/content/drive')

# ==============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# ==============================================================================
FECHA_INICIO = "2024-11-01"
FECHA_FIN = "2024-11-30"

# Ruta de guardado en tu Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Sexto/PRACTICAS_CSIC_IPNA/TFN/"
if not os.path.exists(CARPETA_DRIVE):
    os.makedirs(CARPETA_DRIVE)

ARCHIVO_SALIDA = os.path.join(CARPETA_DRIVE, f"TROPOMI_Tenerife_{FECHA_INICIO[:7].replace('-','_')}.csv")

# Coordenadas de interés (VERTEDERO ARICO TEENRIFE)
LAT_CENTRO, LON_CENTRO = 28.121479, -16.489068 #
MARGEN = 0.4  # Grados de margen alrededor del punto
LAT_MIN, LAT_MAX = LAT_CENTRO - MARGEN, LAT_CENTRO + MARGEN
LON_MIN, LON_MAX = LON_CENTRO - MARGEN, LON_CENTRO + MARGEN

archivo_temporal = "temp_tropomi.nc"

# ==============================================================================
# 2. PROCESAMIENTO DÍA A DÍA
# ==============================================================================
print(f"🚀 Iniciando recolección: {FECHA_INICIO} al {FECHA_FIN}")
dias_rango = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN)
lista_datos_acumulados = []

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

for fecha in dias_rango:
    dia_str = fecha.strftime("%Y-%m-%d")
    print(f"\n📅 Analizando: {dia_str}")

    try:
        # Búsqueda en STAC
        busqueda = catalog.search(
            collections=["sentinel-5p-l2-netcdf"],
            bbox=[LON_MIN, LAT_MIN, LON_MAX, LAT_MAX],
            datetime=f"{dia_str}/{dia_str}"
        )

        items = [item for item in list(busqueda.items()) if "ch4" in item.assets]

        if not items:
            print(f"    ☁️ Sin datos (nubes o sin pasada).")
            continue

        # Tomamos la pasada más relevante
        item = items[0]
        hora_utc = item.datetime
        url = planetary_computer.sign(item).assets["ch4"].href

        # Descarga rápida
        r = requests.get(url, stream=True, timeout=60)
        with open(archivo_temporal, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # Lectura de variables
        ds = xr.open_dataset(archivo_temporal, group="PRODUCT", engine="netcdf4")
        ds_sup = xr.open_dataset(archivo_temporal, group="PRODUCT/SUPPORT_DATA/INPUT_DATA", engine="netcdf4")

        lats = ds.latitude.values[0]
        lons = ds.longitude.values[0]
        ch4_corr = ds.methane_mixing_ratio_bias_corrected.values[0]
        ch4_raw = ds.methane_mixing_ratio.values[0]
        qa = ds.qa_value.values[0]
        aire = ds_sup.dry_air_subcolumns.values[0].sum(axis=-1)

        ds.close(); ds_sup.close(); os.remove(archivo_temporal)

        # Obtención de Viento ERA5
        url_vnt = f"https://archive-api.open-meteo.com/v1/archive?latitude={LAT_CENTRO}&longitude={LON_CENTRO}&start_date={dia_str}&end_date={dia_str}&hourly=wind_speed_10m,wind_direction_10m&wind_speed_unit=ms"
        res_vnt = requests.get(url_vnt).json()
        h_idx = hora_utc.hour
        v_vel = res_vnt['hourly']['wind_speed_10m'][h_idx]
        v_dir = res_vnt['hourly']['wind_direction_10m'][h_idx]

        # Cálculo de componentes
        u10 = -v_vel * np.sin(np.radians(v_dir))
        v10 = -v_vel * np.cos(np.radians(v_dir))

        # --- FILTRADO INTELIGENTE ---
        mask_espacial = (lats >= LAT_MIN) & (lats <= LAT_MAX) & (lons >= LON_MIN) & (lons <= LON_MAX)
        mask_nubes = ~np.isnan(ch4_corr)
        mask = mask_espacial & mask_nubes

        if not np.any(mask):
            print(f"    ⚠️ Día descartado: Todo nubes (NaN).")
            continue

        # DATAFRAME SIN ANOMALÍA
        df_dia = pd.DataFrame({
            'timestamp': hora_utc,
            'lat': lats[mask],
            'lon': lons[mask],
            'metano_ppb': ch4_corr[mask],
            'metano_raw_ppb': ch4_raw[mask],
            'columna_aire': aire[mask],
            'qa_value': qa[mask],
            'u10': u10,
            'v10': v10,
            'v_modulo': v_vel
        })

        if (df_dia['metano_ppb'] == 0).all() or len(df_dia) < 1:
            print(f"    ⚠️ Día descartado por datos corruptos o insuficientes.")
            continue

        lista_datos_acumulados.append(df_dia)
        print(f"    ✅ Éxito: {len(df_dia)} píxeles guardados.")

    except Exception as e:
        print(f"    ❌ Error procesando el día: {e}")

# ==============================================================================
# 3. CONSOLIDACIÓN Y EXPORTACIÓN
# ==============================================================================
if lista_datos_acumulados:
    df_final = pd.concat(lista_datos_acumulados, ignore_index=True)
    df_final.to_csv(ARCHIVO_SALIDA, index=False)
    print("\n" + "="*60)
    print(f"🏁 PROCESO FINALIZADO")
    print(f"📁 Archivo: {ARCHIVO_SALIDA}")
    print(f"📊 Total de píxeles válidos en el mes: {len(df_final)}")
    print("="*60)
else:
    print("\n😭 No se pudo recolectar ningún dato válido para este periodo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Iniciando recolección: 2024-11-01 al 2024-11-30

📅 Analizando: 2024-11-01
    ✅ Éxito: 7 píxeles guardados.

📅 Analizando: 2024-11-02
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-11-03
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-11-04
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-11-05
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-11-06
    ☁️ Sin datos (nubes o sin pasada).

📅 Analizando: 2024-11-07
    ❌ Error procesando el día: The request exceeded the maximum allowed time, please try again. If the issue persists, please contact planetarycomputer@microsoft.com.



📅 Analizando: 2024-11-08
    ⚠️ Día descartado: Todo nubes (NaN).

📅 Analizando: 2024-11-09
    ✅ Éxito: 20 píxeles guardados.

📅 Analizando: 2024-11-10
    ✅ Éxito: 19 píxeles guardados.

📅 Analizando: 2024-11-11
    ✅ Éxito: 10 pí